In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

# Функция mean pooling для усреднения эмбеддингов токенов с маской внимания
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]    # первый элемент - все эмбеддинги токенов
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

# Загрузка модели и токенизатора
tokenizer = AutoTokenizer.from_pretrained("ai-forever/sbert_large_nlu_ru")
model = AutoModel.from_pretrained("ai-forever/sbert_large_nlu_ru")

# Ваш текст (например, full_text из JSONL)
sentences = ["Привет, я - друг Высшей ИТ-школы."]

# Токенизация
encoded_input = tokenizer(sentences, padding=True, truncation=True, max_length=256, return_tensors='pt')

# Вычисление эмбеддингов без градиентов
with torch.no_grad():
    model_output = model(**encoded_input)

# Получение усредненного эмбеддинга для каждого предложения
embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# embeddings — тензор PyTorch, можно преобразовать в numpy и записать в pgvector
embeddings = embeddings.cpu().numpy()

print(embeddings)

[[ 1.0034268   0.2166121  -0.34547028 ... -0.01884868 -1.0223409
  -0.3499685 ]]


In [1]:
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

# --- Mean pooling ---
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # (batch_size, seq_len, hidden_size)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

# --- Модель и токенизатор ---
tokenizer = AutoTokenizer.from_pretrained("ai-forever/sbert_large_nlu_ru")
model = AutoModel.from_pretrained("ai-forever/sbert_large_nlu_ru")
model.eval()

def get_embedding(text):
    encoded_input = tokenizer(text, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        model_output = model(**encoded_input)
    embedding = mean_pooling(model_output, encoded_input["attention_mask"])
    return embedding[0].cpu().numpy().astype(np.float32)  # важно привести к float32, чтобы pgvector корректно принял вектор


with open("./corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        message_number = data.get("message_number")
        full_text = data.get("content", "")
        category = data.get("category")
        subcategory = data.get("subcategory")
        metadata = data.get("metadata", {})

        embedding = get_embedding(full_text)
        print(message_number,
            full_text,
            category,
            subcategory,
            json.dumps(metadata),
            embedding.tolist())


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/863 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

1  Общая информация Приветствие и меню {"tags": ["\u043f\u0440\u0438\u0432\u0435\u0442\u0441\u0442\u0432\u0438\u0435", "\u0433\u043b\u0430\u0432\u043d\u043e\u0435 \u043c\u0435\u043d\u044e"], "updated_at": "2025-07-15T10:00:00+03:00"} [-0.31043022871017456, -0.34128111600875854, -0.4396595358848572, -0.036388009786605835, -0.15227670967578888, -0.4711090624332428, 0.13404053449630737, -0.2252625823020935, -0.20594799518585205, 0.09956075251102448, -0.051587529480457306, 0.0320223867893219, 0.0597788542509079, -0.27013885974884033, -0.11376694589853287, 0.515372097492218, -0.1410665363073349, 0.13455647230148315, 0.03661908209323883, 0.45287585258483887, 1.3919150829315186, -0.25193169713020325, -0.2100217342376709, 0.3541012406349182, -0.5351616740226746, 0.604313313961029, 0.03506774455308914, -0.40216851234436035, 0.5687204003334045, 0.1700822412967682, -0.20100390911102295, -0.7391148805618286, -0.6467406749725342, 0.1821795403957367, 0.6656283140182495, -0.4129667282104492, -0.03492